# Potter Airlines Fare Factors

These functions calculate the five multipliers used by the Potter Airlines pricing formula:

`price = base_fare x time_factor x demand_factor x capacity_factor x seasonal_factor x class_factor`

Each function is independent so that the factors can be tested, explained, and reused by the final pricing logic. 
**All the coefficients here are assumed**

## Pricing method

The model starts with the route's `base_fare` and adjusts it using five transparent business factors. The factor values are project assumptions designed to make fares respond predictably to booking time, demand, aircraft occupancy, travel season, and cabin class. **This description part are AI-generated**

| Factor | Input and pricing rule |
|---|---|
| Time | The multiplier rises from 0.90 for bookings more than 90 days before departure to 1.35 for bookings made 0-3 days before departure. |
| Demand | `route_popularity` is divided into bands from 0.00-1.00. The multiplier ranges from 0.85 for very low popularity to 1.20 for very high popularity. |
| Capacity | Occupancy is calculated as `(capacity - seats_remaining) / capacity`. The multiplier ranges from 0.85 when less than 25% of seats are occupied to 1.45 when at least 95% are occupied. |
| Seasonal | Christmas and New Year use 1.20, summer uses 1.15, March uses 1.10, the January-February low season uses 0.95, and other dates use 1.00. |
| Class | Economy is the default at 1.00. Business class uses 1.75. |

### Why these coefficients were chosen

These coefficients are rule-based assumptions rather than estimates from historical sales. The dataset is synthetic and contains flight characteristics, but it does not contain observed customer purchases or final market prices that could be used to statistically estimate each factor. The selected multipliers therefore create clear and moderate price movements that can be explained from the available fields.

- **Time:** `days_until_departure` ranges from 0 to 151 days. The dataset contains 384 flights more than 90 days from departure, but only 29 flights in the 0-3 day range and 25 in the 4-7 day range. The wider early-booking bands cover the more common observations and provide modest discounts, while the smaller last-minute bands apply progressively larger premiums as departure approaches.
- **Demand:** Almost all `route_popularity` values fall between 0.50 and 0.89, distributed across four similar-sized 0.10 bands. Using 1.00 for the 0.60-0.69 band provides a neutral reference point, with gradual discounts below it and premiums above it.
- **Capacity:** Aircraft occupancy is spread across the full 0%-100% range. Lower-occupancy flights receive discounts to encourage bookings, while the premium increases more sharply above 85% occupancy because available seats have become scarce.
- **Seasonal:** The dataset covers November 2026 through April 2027, including 162 Christmas/New Year flights, 345 January-February low-season flights, 203 March flights, and 334 regular-date flights. These groups support a holiday premium, a post-holiday discount, and a smaller March-break premium. The summer multiplier is retained as a general rule for future data, although the current dataset contains no summer departures.
- **Class:** Economy is the 1.00 baseline, while the 1.75 Business multiplier is a simple project assumption that creates a clear service-level premium without introducing additional unsupported categories.

After all factors are multiplied, the result is bounded between `base_fare * 0.60 * class_factor` and `base_fare * 2.50 * class_factor`, then rounded to two decimal places. The `international` field is retained as flight information but is not a separate multiplier because domestic and international differences are already reflected in the generated base fares.

Relative price bounds are used instead of fixed dollar limits because base fares in the dataset range from $160 to $1,697. Scaling the bounds with both `base_fare` and `class_factor` preserves meaningful differences between short domestic routes, long international routes, Economy fares, and Business fares.

## Flight class

The `Flight` class stores one record from `flights.json`. `Flight.from_dict()` converts a JSON dictionary into a Flight object, `to_dict()` returns the original dictionary structure, and `calculate_price()` produces an Economy or Business quote using the shared pricing functions.

The **@classmethod** converts each flight dictionary loaded from the JSON dataset directly into a Flight object, avoiding the need to pass every attribute manually.

The examples at the end of the notebook load one domestic flight and one international flight from the dataset. The tested quotes are PA1000 Vancouver-Calgary at $152.26 Economy and $266.45 Business, and PA2000 Vancouver-Los Angeles at $316.01 Economy and $553.01 Business.

In [ ]:
def calculate_time_factor(days_until_departure):
    #Return a fare multiplier based on how soon the flight departs.
    if days_until_departure <= 3:
        return 1.35
    if days_until_departure <= 7:
        return 1.25
    if days_until_departure <= 14:
        return 1.15
    if days_until_departure <= 30:
        return 1.08
    if days_until_departure <= 60:
        return 1.00
    if days_until_departure <= 90:
        return 0.95
    return 0.90

In [ ]:
def calculate_demand_factor(route_popularity):
    #Return a fare multiplier based on route popularity.
    if route_popularity < 0.50:
        return 0.85
    if route_popularity < 0.60:
        return 0.90
    if route_popularity < 0.70:
        return 1.00
    if route_popularity < 0.80:
        return 1.08
    if route_popularity < 0.90:
        return 1.15
    return 1.20

In [ ]:
def calculate_capacity_factor(seats_remaining, capacity):
    #Return a fare multiplier based on the proportion of seats already sold.
    occupancy_rate = (capacity - seats_remaining) / capacity

    if occupancy_rate < 0.25:
        return 0.85
    if occupancy_rate < 0.50:
        return 0.95
    if occupancy_rate < 0.70:
        return 1.05
    if occupancy_rate < 0.85:
        return 1.15
    if occupancy_rate < 0.95:
        return 1.30
    return 1.45

In [ ]:
from datetime import datetime


def calculate_seasonal_factor(departure_date):
    #Return a fare multiplier for holiday, summer, spring-break, or regular travel.
    departure = datetime.strptime(departure_date, "%m-%d-%Y")
    month = departure.month
    day = departure.day

    if (month == 12 and day >= 15) or (month == 1 and day <= 5):
        return 1.20
    if month in (6, 7, 8):
        return 1.15
    if month == 3:
        return 1.10
    if month in (1, 2):
        return 0.95
    return 1.00

In [21]:
CLASS_FACTORS = {
    "economy": 1.00,
    "business": 1.75,
}


def calculate_class_factor(fare_class="economy"):
    """Return the multiplier for economy or business class."""
    return CLASS_FACTORS[fare_class.strip().lower()]

In [ ]:
def calculate_final_price(flight, fare_class="economy"):
    #Combine all fare factors and return the bounded final price.
    base_fare = flight["base_fare"]

    time_factor = calculate_time_factor(flight["days_until_departure"])
    demand_factor = calculate_demand_factor(flight["route_popularity"])
    capacity_factor = calculate_capacity_factor(
        flight["seats_remaining"],
        flight["capacity"],)
    seasonal_factor = calculate_seasonal_factor(flight["departure_date"])
    class_factor = calculate_class_factor(fare_class)

    raw_price = (base_fare*time_factor*demand_factor*capacity_factor*seasonal_factor*class_factor)

    minimum_fare = base_fare * 0.60 * class_factor
    maximum_fare = base_fare * 2.50 * class_factor
    final_price = max(minimum_fare, min(raw_price, maximum_fare))

    return round(final_price, 2)

In [ ]:
class Flight:
    #Represent one Potter Airlines flight and calculate its fare.

    def __init__(
        self,
        flight_id,
        origin,
        destination,
        departure_date,
        days_until_departure,
        base_fare,
        seats_remaining,
        capacity,
        route_popularity,
        international,
    ):
        self.flight_id = flight_id
        self.origin = origin
        self.destination = destination
        self.departure_date = departure_date
        self.days_until_departure = days_until_departure
        self.base_fare = base_fare
        self.seats_remaining = seats_remaining
        self.capacity = capacity
        self.route_popularity = route_popularity
        self.international = international

    @classmethod 
    def from_dict(cls, flight_data):
        #Create a Flight object from one flight dictionary.
        return cls(
            flight_id=flight_data["flight_id"],
            origin=flight_data["origin"],
            destination=flight_data["destination"],
            departure_date=flight_data["departure_date"],
            days_until_departure=flight_data["days_until_departure"],
            base_fare=flight_data["base_fare"],
            seats_remaining=flight_data["seats_remaining"],
            capacity=flight_data["capacity"],
            route_popularity=flight_data["route_popularity"],
            international=flight_data["international"],
        )
    ##easier to convert JSON flight records into consistent Flight objects that can be reused later for pricing, 
    #database operations, filtering, and system integration.

    def to_dict(self):
        """Return the flight information in the original dictionary format."""
        return {
            "flight_id": self.flight_id,
            "origin": self.origin,
            "destination": self.destination,
            "departure_date": self.departure_date,
            "days_until_departure": self.days_until_departure,
            "base_fare": self.base_fare,
            "seats_remaining": self.seats_remaining,
            "capacity": self.capacity,
            "route_popularity": self.route_popularity,
            "international": self.international,
        }

    def calculate_price(self, fare_class="economy"):
        """Calculate this flight's final economy or business fare."""
        return calculate_final_price(self.to_dict(), fare_class)

In [24]:
import json
from pathlib import Path


data_paths = [
    Path("data/flights.json"),
    Path("mma6-potter-airlines/data/flights.json"),
]
data_path = next(path for path in data_paths if path.exists())

with data_path.open("r", encoding="utf-8") as file:
    flight_records = json.load(file)

domestic_data = next(record for record in flight_records if record["international"] == 0)
international_data = next(record for record in flight_records if record["international"] == 1)

domestic_flight = Flight.from_dict(domestic_data)
international_flight = Flight.from_dict(international_data)

In [25]:
domestic_quote = {
    "flight_data": domestic_flight.to_dict(),
    "economy_price": domestic_flight.calculate_price(),
    "business_price": domestic_flight.calculate_price("business"),
}

domestic_quote

{'flight_data': {'flight_id': 'PA1000',
  'origin': 'Vancouver',
  'destination': 'Calgary',
  'departure_date': '02-28-2027',
  'days_until_departure': 119,
  'base_fare': 163,
  'seats_remaining': 57,
  'capacity': 91,
  'route_popularity': 0.87,
  'international': 0},
 'economy_price': 152.26,
 'business_price': 266.45}

In [26]:
international_quote = {
    "flight_data": international_flight.to_dict(),
    "economy_price": international_flight.calculate_price(),
    "business_price": international_flight.calculate_price("business"),
}

international_quote

{'flight_data': {'flight_id': 'PA2000',
  'origin': 'Vancouver',
  'destination': 'Los Angeles',
  'departure_date': '12-02-2026',
  'days_until_departure': 31,
  'base_fare': 308,
  'seats_remaining': 93,
  'capacity': 173,
  'route_popularity': 0.78,
  'international': 1},
 'economy_price': 316.01,
 'business_price': 553.01}